# SignaCulture: mT5 French → Belgian LSFB, with Google Drive checkpoints
Select **Runtime → Change runtime type → T4 GPU**, then run cells in order. This is an exploratory weak-alignment experiment, **not France-LSF translation**. The Dicta-Sign motion library is separate. No SSH or paid remote-control setup needed.

Uses official **training shard 000002**, never trains on official test shard 000000. Candidates are split by signer where possible, otherwise recording. This internal holdout is not the official benchmark. First run: 30 steps; change TRAIN_STEPS to 100 for a separate larger run after examining losses and predictions. Mount your own Drive interactively; no credentials go into this notebook.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys, json, shutil
DRIVE = Path('/content/drive/MyDrive/SignaCulture/mt5-lsfb-poc')
DRIVE.mkdir(parents=True, exist_ok=True)
TRAIN_STEPS = 30
ALLOW_WEAK_LABELS = True  # Explicit exploratory labels; no human approval is implied.
RUN = DRIVE / f'mt5-small-steps-{TRAIN_STEPS}'
WORK = Path('/content/signaculture-lsfb'); WORK.mkdir(exist_ok=True)
print('Persistent checkpoints/model/results:', RUN)
print('Allow several GB of free Drive space; do not share private annotations or model weights automatically.')


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.57.6', 'accelerate==1.15.0', 'sentencepiece==0.2.3', 'protobuf>=4.25', 'huggingface-hub==0.36.2'], check=True)
import torch
assert torch.cuda.is_available(), 'Choose a GPU runtime before training.'
print(torch.cuda.get_device_name(), torch.__version__)
subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=(DRIVE/'environment.txt').open('w'), check=True)


In [ ]:
SCRIPT = WORK / "lsfb_poc.py"
SCRIPT.write_text('"""Small, recording/signer-disjoint French → Belgian LSFB experiment."""\nimport argparse, csv, hashlib, json, math, random, tarfile\nfrom pathlib import Path\n\nPREFIX = \'translate French to LSFB glosses: \'\nREPO = \'ppoitier/lsfb-cont\'\n\ndef dump(path, value):\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2)+\'\\n\')\n\ndef extract(archive, output, limit=300, overlap=.6):\n    recordings = {}\n    with tarfile.open(archive) as tar:\n        for member in tar:\n            suffix = next((s for s in [\'annotations.translation.json\', \'annotations.both_hands.json\', \'signer.txt\'] if member.name.endswith(\'.\'+s)), None)\n            if not suffix or not member.isfile(): continue\n            if member.size > 20_000_000: raise ValueError(\'Unexpectedly large annotation\')\n            key = member.name[:-(len(suffix)+1)]\n            value = tar.extractfile(member).read().decode(\'utf-8\')\n            recordings.setdefault(key, {})[suffix] = value.strip() if suffix==\'signer.txt\' else json.loads(value)\n    rows=[]; seen=set()\n    for recording, data in sorted(recordings.items()):\n        translations=data.get(\'annotations.translation.json\',[])\n        signs=data.get(\'annotations.both_hands.json\',[])\n        if not isinstance(translations,list) or not isinstance(signs,list): raise ValueError(\'Unrecognized corpus JSON: expected segment lists\')\n        for t in translations:\n            start,end=float(t[\'start_ms\']),float(t[\'end_ms\'])\n            if end<=start or not t.get(\'text\',\'\').strip(): continue\n            events=set()\n            for s in signs:\n                a,b=float(s[\'start_ms\']),float(s[\'end_ms\']); gloss=s.get(\'gloss\')\n                if b>a and isinstance(gloss,str) and gloss.strip() and max(0,min(end,b)-max(start,a))/(b-a)>=overlap:\n                    events.add((a,b,gloss.strip()))\n            target=\' \'.join(g for a,b,g in sorted(events))\n            text=\' \'.join(t[\'text\'].split())\n            # Exact repeated French text is removed globally, even when targets differ.\n            if not target or text.casefold() in seen: continue\n            seen.add(text.casefold())\n            rows.append(dict(id=f\'{recording}:{start}:{end}\',recording_id=recording,signer_id=data.get(\'signer.txt\'),language=\'LSFB\',french_text=text,gloss_sequence=target,source=REPO,annotation_tier=\'both_hands\',review_status=\'unreviewed_auto_alignment\',start_ms=start,end_ms=end))\n    # Sample across recordings before capping; never take only the first recording.\n    random.Random(42).shuffle(rows); rows=rows[:limit]\n    Path(output).parent.mkdir(parents=True,exist_ok=True)\n    Path(output).write_text(\'\'.join(json.dumps(r,ensure_ascii=False)+\'\\n\' for r in rows))\n    report=dict(candidates=len(rows),recordings=len({r[\'recording_id\'] for r in rows}),signers=sorted({r[\'signer_id\'] for r in rows if r[\'signer_id\']}),archive_sha256=hashlib.file_digest(open(archive,\'rb\'),\'sha256\').hexdigest(),alignment=\'weak temporal overlap; simultaneous signs flattened\',license=\'CC-BY-NC-SA-4.0\')\n    dump(str(output)+\'.audit.json\',report); return report\n\ndef split(source, folder, allow_weak=False):\n    rows=[json.loads(l) for l in Path(source).read_text().splitlines() if l.strip()]\n    rows=[r for r in rows if r[\'review_status\']==\'approved\' or allow_weak and r[\'review_status\']==\'unreviewed_auto_alignment\']\n    if not rows: raise ValueError(\'No eligible examples. Review candidates or explicitly enable weak experimental labels.\')\n    signer_groups={r.get(\'signer_id\') for r in rows}\n    group=\'signer_id\' if None not in signer_groups and len(signer_groups)>=3 else \'recording_id\'\n    groups=sorted({r[group] for r in rows});random.Random(42).shuffle(groups)\n    if len(groups)<3: raise ValueError(\'Fewer than three independent groups: obtain another training shard\')\n    n=max(1,round(.15*len(groups))); selected={\'test\':set(groups[:n]),\'val\':set(groups[n:2*n]),\'train\':set(groups[2*n:])}\n    folder=Path(folder);folder.mkdir(parents=True,exist_ok=True);report={\'grouping\':group,\'weak_supervision\':allow_weak,\'evaluation_scope\':\'internal held-out subset of official training shard, not official benchmark\',\'splits\':{}}\n    for name, ids in selected.items():\n        part=[r for r in rows if r[group] in ids]\n        if not part: raise ValueError(\'Empty split\')\n        path=folder/(name+\'.jsonl\');path.write_text(\'\'.join(json.dumps(r,ensure_ascii=False)+\'\\n\' for r in part))\n        report[\'splits\'][name]={\'examples\':len(part),\'groups\':sorted(ids),\'sha256\':hashlib.sha256(path.read_bytes()).hexdigest()}\n    dump(folder/\'split_report.json\',report);return report\n\ndef distance(a,b):\n    prev=list(range(len(b)+1))\n    for i,x in enumerate(a,1):\n        cur=[i]\n        for j,y in enumerate(b,1):cur.append(min(cur[-1]+1,prev[j]+1,prev[j-1]+(x!=y)))\n        prev=cur\n    return prev[-1]\n\ndef train(data, output, steps=30, base=\'google/mt5-small\', revision=None, max_length=96, eval_every=0):\n    import torch\n    from torch.utils.data import Dataset\n    from transformers import AutoTokenizer,AutoModelForSeq2SeqLM,DataCollatorForSeq2Seq,Seq2SeqTrainer,Seq2SeqTrainingArguments,TrainerCallback,set_seed\n    set_seed(42);data=Path(data);output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    report=json.loads((data/\'split_report.json\').read_text())\n    for name, entry in report[\'splits\'].items():\n        if hashlib.sha256((data/(name+\'.jsonl\')).read_bytes()).hexdigest()!=entry[\'sha256\']:raise ValueError(\'Split changed since freezing\')\n    fingerprint=hashlib.sha256((data/\'split_report.json\').read_bytes()).hexdigest()\n    manifest={\'base\':base,\'revision\':revision,\'split_sha256\':fingerprint,\'steps\':steps,\'language\':\'LSFB\',\'weak_supervision\':report[\'weak_supervision\'],\'precision\':\'bf16 when supported; otherwise fp32, never fp16\'}\n    if max_length!=96:manifest[\'max_length\']=max_length\n    if eval_every:manifest[\'eval_every\']=eval_every\n    plan=output/\'run_config.json\'\n    if plan.exists() and json.loads(plan.read_text())!=manifest:raise ValueError(\'Different configuration: use a new Drive run folder\')\n    dump(plan,manifest)\n    if (output/\'COMPLETED.json\').is_file() and all((output/name).is_file() for name in [\'final/model.safetensors\',\'final/config.json\',\'final/tokenizer_config.json\',\'metrics.json\']):\n        print(\'Run already complete; preserving the saved model and results.\',flush=True)\n        return\n    tok=AutoTokenizer.from_pretrained(base,revision=revision,use_fast=False)\n    model=AutoModelForSeq2SeqLM.from_pretrained(base,revision=revision);model.config.use_cache=False\n    class Pairs(Dataset):\n        def __init__(self,name):\n            self.rows=[json.loads(l) for l in (data/(name+\'.jsonl\')).read_text().splitlines()]\n            self.items=[]\n            for r in self.rows:\n                item=tok(PREFIX+r[\'french_text\'],return_token_type_ids=False);item[\'labels\']=tok(text_target=r[\'gloss_sequence\'])[\'input_ids\']\n                if max(len(item[\'input_ids\']),len(item[\'labels\']))>max_length: raise ValueError(\'Overlength example: review or segment it; no silent target truncation\')\n                self.items.append(item)\n        def __len__(self): return len(self.items)\n        def __getitem__(self,i): return self.items[i]\n    class Finite(TrainerCallback):\n        def on_log(self,args,state,control,logs=None,**kwargs):\n            if any(isinstance(v,(float,int)) and not math.isfinite(v) for v in (logs or {}).values()): raise ValueError(\'Nonfinite training log\')\n    interval=eval_every or max(1,steps//2)\n    args=Seq2SeqTrainingArguments(output_dir=str(output/\'checkpoints\'),max_steps=steps,per_device_train_batch_size=1,per_device_eval_batch_size=1,gradient_accumulation_steps=4,learning_rate=5e-5,optim=\'adafactor\',gradient_checkpointing=True,fp16=False,bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),eval_strategy=\'steps\',eval_steps=interval,save_steps=interval,save_total_limit=2,load_best_model_at_end=True,metric_for_best_model=\'eval_loss\',logging_steps=1,logging_nan_inf_filter=False,report_to=\'none\',seed=42)\n    trainer=Seq2SeqTrainer(model=model,args=args,train_dataset=Pairs(\'train\'),eval_dataset=Pairs(\'val\'),data_collator=DataCollatorForSeq2Seq(tok,model=model),processing_class=tok,callbacks=[Finite()])\n    # A disconnect can leave a half-written directory on Drive. Resume only complete saves.\n    checkpoints=[]\n    for checkpoint in (output/\'checkpoints\').glob(\'checkpoint-*\'):\n        if all((checkpoint/name).is_file() for name in [\'trainer_state.json\',\'optimizer.pt\',\'scheduler.pt\']) and ((checkpoint/\'model.safetensors\').is_file() or (checkpoint/\'model.safetensors.index.json\').is_file()):\n            checkpoints.append(checkpoint)\n    latest=str(max(checkpoints,key=lambda p:int(p.name.split(\'-\')[-1]))) if checkpoints else None\n    trainer.train(resume_from_checkpoint=latest)\n    trainer.save_model(str(output/\'final\'));tok.save_pretrained(output/\'final\');dump(output/\'history.json\',trainer.state.log_history)\n    # Test is decoded only after validation checkpoint selection.\n    from difflib import SequenceMatcher\n    model.eval();model.config.use_cache=True;rows=Pairs(\'test\').rows;training=Pairs(\'train\').rows;predictions=[];refs=[];baseline=[]\n    torch.set_num_threads(2)\n    for begin in range(0,len(rows),8):\n        part=rows[begin:begin+8]\n        with torch.inference_mode():\n            x=tok([PREFIX+r[\'french_text\'] for r in part],padding=True,return_tensors=\'pt\',return_token_type_ids=False).to(model.device)\n            outputs=model.generate(**x,max_new_tokens=max_length,num_beams=4)\n        for r,prediction in zip(part,tok.batch_decode(outputs,skip_special_tokens=True)):\n            predictions.append({\'french\':r[\'french_text\'],\'reference\':r[\'gloss_sequence\'],\'prediction\':prediction,\'recording_id\':r[\'recording_id\']});refs.append(r[\'gloss_sequence\'])\n            baseline.append(max(training,key=lambda t:SequenceMatcher(None,t[\'french_text\'].casefold(),r[\'french_text\'].casefold()).ratio())[\'gloss_sequence\'])\n        dump(output/\'predictions.partial.json\',predictions)\n        print(f\'Evaluated {len(predictions)}/{len(rows)} held-out examples\',flush=True)\n    def score(pred):return {\'exact_match\':sum(a==b for a,b in zip(refs,pred))/len(refs),\'wer\':sum(distance(a.split(),b.split()) for a,b in zip(refs,pred))/sum(len(a.split()) for a in refs)}\n    dump(output/\'predictions.json\',predictions);dump(output/\'metrics.json\',{\'model\':score([r[\'prediction\'] for r in predictions]),\'train_only_retrieval\':score(baseline),\'examples\':len(rows),\'official_benchmark\':False,\'linguistic_validation\':False})\n    dump(output/\'COMPLETED.json\',{\'steps\':trainer.state.global_step,\'best_checkpoint\':trainer.state.best_model_checkpoint,\'final_model\':\'final\',\'target_language\':\'Belgian LSFB; not France LSF\'})\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();sub=p.add_subparsers(dest=\'cmd\',required=True)\n    a=sub.add_parser(\'extract\');a.add_argument(\'archive\');a.add_argument(\'output\');a.add_argument(\'--limit\',type=int,default=300)\n    a=sub.add_parser(\'split\');a.add_argument(\'input\');a.add_argument(\'output\');a.add_argument(\'--allow-weak\',action=\'store_true\')\n    a=sub.add_parser(\'train\');a.add_argument(\'data\');a.add_argument(\'output\');a.add_argument(\'--steps\',type=int,default=30);a.add_argument(\'--base\',default=\'google/mt5-small\');a.add_argument(\'--revision\');a.add_argument(\'--max-length\',type=int,default=96);a.add_argument(\'--eval-every\',type=int,default=0)\n    a=p.parse_args()\n    if a.cmd==\'extract\':print(extract(a.archive,a.output,a.limit))\n    elif a.cmd==\'split\':print(split(a.input,a.output,a.allow_weak))\n    else:train(a.data,a.output,a.steps,a.base,a.revision,a.max_length,a.eval_every)\n', encoding="utf-8")
print(SCRIPT)


In [ ]:
from huggingface_hub import hf_hub_download
DATA_REVISION = '64b0c112f59d3710367fb736ef1808f37ff15e28'
MODEL_REVISION = '73fb5dbe4756edadc8fbe8c769b0a109493acf7a'
archive = hf_hub_download('ppoitier/lsfb-cont', 'shards/annotated/shard_000002.tar', repo_type='dataset', revision=DATA_REVISION, local_dir=str(WORK/'raw'))
subprocess.run([sys.executable, str(SCRIPT), 'extract', archive, str(DRIVE/'candidates.jsonl'), '--limit', '300'], check=True)
(DRIVE/'source.json').write_text(json.dumps({'repository':'ppoitier/lsfb-cont','revision':DATA_REVISION,'shard':2,'official_split':'train','license':'CC-BY-NC-SA-4.0'},indent=2))
print((DRIVE/'candidates.jsonl.audit.json').read_text())


## Inspect candidates before training
These references come from timestamp overlap, not validated translations. Inspect the sample below; edit `candidates.jsonl` in Drive to remove bad alignments or mark reviewed examples `approved`. To require reviewed examples, set `ALLOW_WEAK_LABELS=False`. No poses or GAN are trained. If annotations are unusable, stop and improve the data.


In [ ]:
import random
rows = [json.loads(line) for line in (DRIVE/'candidates.jsonl').read_text().splitlines()]
for row in random.Random(42).sample(rows, min(30,len(rows))):
    print(row['recording_id'], row['signer_id'], '\nFR:', row['french_text'], '\nLSFB:',row['gloss_sequence'], '\n')


In [ ]:
from transformers import AutoTokenizer
length_tokenizer=AutoTokenizer.from_pretrained('google/mt5-small',revision=MODEL_REVISION,use_fast=False)
path=DRIVE/'candidates.jsonl'
rows=[json.loads(line) for line in path.read_text().splitlines() if line.strip()]
kept=[];rejected=[]
for r in rows:
    n=max(len(length_tokenizer('translate French to LSFB glosses: '+r['french_text'])['input_ids']),len(length_tokenizer(text_target=r['gloss_sequence'])['input_ids']))
    (kept if n<=96 else rejected).append(r)
(DRIVE/'overlength-candidates.json').write_text(json.dumps(rejected,ensure_ascii=False,indent=2))
path.write_text(''.join(json.dumps(r,ensure_ascii=False)+'\n' for r in kept))
print('Retained:',len(kept),'Overlength (retained separately):',len(rejected))


In [ ]:
command=[sys.executable,str(SCRIPT),'split',str(DRIVE/'candidates.jsonl'),str(DRIVE/'splits')]
if ALLOW_WEAK_LABELS: command.append('--allow-weak')
subprocess.run(command,check=True)
print((DRIVE/'splits/split_report.json').read_text())


## Train and persist directly to Drive
FP16 is disabled. Adafactor and gradient checkpointing are enabled. Precision follows PyTorch capability detection: BF16 when supported (which newer runtimes can emulate on T4), otherwise FP32. The actual choice is saved in training_args.bin. Checkpoints include optimizer/trainer state every half-run; rerunning this cell resumes the latest complete checkpoint. If you change data/configuration, use a new run directory. Test predictions are generated after validation-based checkpoint selection.


In [ ]:
subprocess.run([sys.executable,str(SCRIPT),'train',str(DRIVE/'splits'),str(RUN),'--steps',str(TRAIN_STEPS),'--revision',MODEL_REVISION],check=True)
assert (RUN/'COMPLETED.json').exists()
print((RUN/'metrics.json').read_text())
for example in json.loads((RUN/'predictions.json').read_text())[:10]: print(example)
print('Saved model/tokenizer:', RUN/'final')


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
final=RUN/'final'
tokenizer=AutoTokenizer.from_pretrained(final,use_fast=False)
model=AutoModelForSeq2SeqLM.from_pretrained(final).to('cuda').eval()
text='Bonjour, comment allez-vous ?'  # Unseen input: output may be poor on this small PoC.
with torch.no_grad():
    inputs=tokenizer('translate French to LSFB glosses: '+text,return_tensors='pt').to('cuda')
    print(tokenizer.decode(model.generate(**inputs,max_new_tokens=96,num_beams=4)[0],skip_special_tokens=True))


## Saved artifacts and interpretation
Drive contains `candidates.jsonl`, extraction audit and source revision; frozen `splits/`; and each run's `checkpoints/`, `final/` model+tokenizer, `history.json`, `metrics.json`, `predictions.json`, `run_config.json` and `COMPLETED.json`. A completed run is not proof of usable translation. Compare against train-only retrieval; inspect empty/repetitive outputs. No upload/publishing happens.

Sources: [LSFB-CONT](https://huggingface.co/datasets/ppoitier/lsfb-cont), [mT5](https://huggingface.co/docs/transformers/model_doc/mt5). CC BY-NC-SA 4.0 source obligations apply; keep this experiment private while reviewing derivative terms.
